# 02 · 混合精度、梯度累积与分布式训练  Mixed Precision, Grad Accumulation & DDP

> **能力标签**：GA4（训练优化 / Training Optimization）

## 目标 Objectives

完成本课后，你应该能够：

1. 解释**混合精度训练**（mixed-precision training）：为何用 `bfloat16`/`float16` 加速前向计算，同时保留 `float32` 参数更新；使用 `torch.autocast` 实现。
2. 理解 `GradScaler` 的作用：在 `float16` 模式下防止梯度下溢（underflow），以及 `bfloat16` 为何通常不需要它。
3. 实现**梯度累积**（gradient accumulation）：用多个小批次（microbatch）模拟大批次（large batch），对应 `w10-grad-accum` 练习。
4. 描述 **DDP（DistributedDataParallel）**的原理：进程（ranks）、`all-reduce` 梯度同步、分片验证（sharded validation）。
5. 在纯 PyTorch 中运行可验证的 `torch.autocast` CPU 示例（`bfloat16`）。

> 对应能力：**GA4**

## 概念讲解 Concepts

### 1. 混合精度训练 Mixed-Precision Training

**动机**：GPU 的 Tensor Core 对 `float16`/`bfloat16` 计算吞吐量远高于 `float32`（通常 2-8×），同时显存占用减半。

**核心思路**：

| 阶段 | 精度 | 原因 |
|------|------|------|
| 前向传播 Forward | `bfloat16` / `float16` | 速度快、显存省 |
| 损失计算 Loss | `bfloat16` / `float16` | 同上 |
| 反向传播 Backward | `bfloat16` / `float16` | 梯度在低精度下计算 |
| 参数更新 Optimizer step | `float32` | 数值稳定性要求高 |

`torch.autocast` 会自动决定哪些算子用低精度，哪些保持 `float32`。

---

### 2. torch.autocast 用法

```python
with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    y = model(x)
    loss = loss_fn(y, target)
loss.backward()      # 反向传播在 autocast 外
optimizer.step()
```

- `device_type="cpu"` 也支持（CPU 上 `bfloat16` 在 torch 2.x 可用）。
- `bfloat16`：16 位浮点，与 `float32` 同样的指数位数（8 bit），动态范围大，**通常不需要 GradScaler**。
- `float16`：精度高但动态范围小，需要 `GradScaler` 防止梯度下溢。

---

### 3. GradScaler (float16 专用)

```python
# 伪代码
scaler = torch.cuda.amp.GradScaler()    # 仅 float16 需要

with torch.autocast(device_type="cuda", dtype=torch.float16):
    loss = model(x)

scaler.scale(loss).backward()           # loss * scale_factor 再反向
scaler.step(optimizer)                  # 自动 unscale 后更新
scaler.update()                         # 调整 scale_factor
```

`GradScaler` 的逻辑：将损失乘以大系数（scale factor）再反向，避免 `float16` 梯度变 0；优化器步骤前自动除回去。

---

### 4. 梯度累积 Gradient Accumulation

**问题**：显存不足时无法使用大 batch（如 batch_size=512）。
**解决**：将大 batch 拆成 $K$ 个 microbatch，分别前向+反向，最后统一 `optimizer.step()`。

关键：每个 microbatch 的损失要除以 $K$（归一化），使梯度等价于大 batch：

$$\text{loss}_{\text{micro}} = \frac{\text{loss}(\text{microbatch})}{K}$$

对应 `w10-grad-accum` 练习的 `accumulate_and_step` 函数。

> **注意**：此等价于全批次梯度仅对**等大小**的 microbatch 成立；若各批次大小不同，需按样本数加权归一化。

---

### 5. DDP 分布式数据并行 Distributed Data Parallel

**DDP** 让多个 GPU（或节点）各持有模型副本，分别处理不同数据，训练后同步梯度：

```
GPU 0: microbatch A -> loss_A -> grad_A --+
GPU 1: microbatch B -> loss_B -> grad_B --+--> all-reduce --> 平均梯度 --> optimizer.step()
GPU 2: microbatch C -> loss_C -> grad_C --+
```

**核心操作**：
- **`all-reduce`**：所有 rank 交换梯度并取平均，每个 rank 最终拥有相同的梯度。
- **分片验证**（sharded validation）：验证集按 rank 分片，避免重复计算；结果用 `dist.all_gather` 汇总。

**启动方式**（伪代码，不可直接运行）：

```bash
# 启动 2 个 GPU 的 DDP 训练
torchrun --nproc_per_node=2 train.py
```

```python
# 伪代码
dist.init_process_group("nccl")
model = DDP(model, device_ids=[local_rank])
# 其余训练循环与单卡完全相同
```

**本课不导入 `torch.distributed`**（多进程环境复杂）；我们在示例区演示梯度累积的纯 torch 实现。

## 示例 Worked Example

In [ ]:
# Setup — 自包含，CPU，可复现
import tempfile, os
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
DEVICE = torch.device("cpu")
TMPDIR = tempfile.mkdtemp()
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

In [ ]:
# 1. torch.autocast CPU bfloat16 示例
# 这是本笔记本可运行的 autocast 演示 (CPU bf16 在 torch 2.x 支持)

torch.manual_seed(7)
model_fp32 = nn.Sequential(
    nn.Linear(8, 16), nn.ReLU(),
    nn.Linear(16, 4),
)

x = torch.randn(4, 8)                  # float32 输入

# 不用 autocast (float32)
with torch.no_grad():
    y_fp32 = model_fp32(x)
print(f"float32 输出 dtype: {y_fp32.dtype}  shape: {y_fp32.shape}")

# 用 autocast (bfloat16 on CPU)
with torch.no_grad(), torch.autocast(device_type="cpu", dtype=torch.bfloat16):
    y_bf16 = model_fp32(x)
print(f"bfloat16 输出 dtype: {y_bf16.dtype}  shape: {y_bf16.shape}")

# 数值应相近（bf16 精度略低但动态范围与 fp32 相同）
err = (y_fp32 - y_bf16.float()).abs().max().item()
print(f"最大误差 max abs error (fp32 vs bf16): {err:.4f}")
assert err < 0.1, "误差过大，请检查 autocast 用法"
print("\u2713 torch.autocast CPU bfloat16 通过 passed")

In [ ]:
# 2. autocast 在训练步骤中的用法 autocast in training step
# 演示 forward + loss 在 autocast 内，backward 在外

torch.manual_seed(42)
model2 = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
optimizer2 = torch.optim.SGD(model2.parameters(), lr=0.01)

X2 = torch.randn(16, 4)
y2 = torch.randn(16, 1)

loss_vals = []
for step in range(5):
    optimizer2.zero_grad()

    # forward + loss 在 autocast 内
    with torch.autocast(device_type="cpu", dtype=torch.bfloat16):
        pred = model2(X2)
        loss = F.mse_loss(pred, y2)

    # backward 在 autocast 外（梯度用 fp32 计算）
    loss.backward()
    optimizer2.step()
    loss_vals.append(loss.item())

print("Loss 历史 loss history:", [f"{v:.4f}" for v in loss_vals])
assert loss_vals[-1] < loss_vals[0], "损失应递减"
print("\u2713 autocast 训练步骤通过 autocast training step passed")

In [ ]:
# 3. 梯度累积实现 Gradient accumulation (mirrors w10-grad-accum)

def accumulate_and_step(model, optimizer, microbatches, loss_fn):
    '''梯度累积：对每个 microbatch 前向+反向累积梯度（损失按 microbatch 数归一化），
    最后 step 一次。返回累积的平均损失（float）。
    Gradient accumulation: forward + backward per microbatch with normalized loss,
    then one optimizer step. Returns mean loss (float).
    mirrors the w10-grad-accum exercise solution.
    '''
    optimizer.zero_grad()
    n = len(microbatches)
    total = 0.0
    for xb, yb in microbatches:
        loss = loss_fn(model(xb), yb) / n     # 归一化关键 normalization key
        loss.backward()
        total += float(loss)
    optimizer.step()
    return total


# 验证：grad-accum 等价于 full-batch step
torch.manual_seed(0)
X_val = torch.randn(8, 3); Y_val = torch.randn(8, 1)
loss_fn_val = nn.MSELoss()

def make_model():
    torch.manual_seed(1)
    return nn.Linear(3, 1)

# full batch (1 microbatch of all 8)
m_full = make_model(); opt_full = torch.optim.SGD(m_full.parameters(), lr=0.1)
accumulate_and_step(m_full, opt_full, [(X_val, Y_val)], loss_fn_val)

# accumulated (2 microbatches of 4)
m_acc = make_model(); opt_acc = torch.optim.SGD(m_acc.parameters(), lr=0.1)
accumulate_and_step(m_acc, opt_acc, [(X_val[:4], Y_val[:4]), (X_val[4:], Y_val[4:])], loss_fn_val)

# 参数应完全相同 params should be identical
for (n1, p1), (n2, p2) in zip(m_full.named_parameters(), m_acc.named_parameters()):
    assert torch.allclose(p1, p2, atol=1e-6), f"{n1}: grad-accum != full-batch"

print("\u2713 梯度累积等价于全批次 grad-accum == full-batch step passed")

## 动手 Exercise

在下面的 cell 中，**实现 `mixed_precision_train_step`**：

- 接受 `model`、`optimizer`、`xb`、`yb` 参数
- 使用 `torch.autocast(device_type="cpu", dtype=torch.bfloat16)` 包裹 forward + loss
- backward 在 autocast 外
- 返回 loss 值（float）

In [ ]:
def mixed_precision_train_step(model, optimizer, xb, yb):
    '''混合精度训练单步（CPU bfloat16）。
    Mixed-precision training step (CPU bfloat16).
    '''
    # TODO: 实现
    # Hint:
    #   optimizer.zero_grad()
    #   with torch.autocast(device_type="cpu", dtype=torch.bfloat16):
    #       pred = model(xb)
    #       loss = F.mse_loss(pred, yb)
    #   loss.backward()
    #   optimizer.step()
    #   return loss.item()
    raise NotImplementedError("请实现 mixed_precision_train_step")

# 实现后取消注释测试
# torch.manual_seed(0)
# m_ex = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
# opt_ex = torch.optim.Adam(m_ex.parameters(), lr=1e-3)
# xb_ex = torch.randn(8, 4); yb_ex = torch.randn(8, 1)
# loss_val = mixed_precision_train_step(m_ex, opt_ex, xb_ex, yb_ex)
# print(f"Loss: {loss_val:.4f}")
# assert isinstance(loss_val, float) and loss_val > 0
# print("\u2713 mixed_precision_train_step 通过 passed")
print("提示：实现后取消注释运行验证")

## 延伸阅读 Further Reading

1. **PyTorch 混合精度训练指南**：<https://pytorch.org/docs/stable/amp.html> — `torch.autocast` 和 `GradScaler` 完整文档。
2. **bfloat16 vs float16**：Google Brain 的 bfloat16 设计；NVIDIA 关于 Tensor Core 精度对比。
3. **PyTorch DDP 教程**：<https://pytorch.org/tutorials/intermediate/ddp_tutorial.html> — 官方 DDP 入门，包含多机多卡示例。
4. **`torchrun` 启动器**：<https://pytorch.org/docs/stable/elastic/run.html> — 替代 `torch.distributed.launch`，弹性训练。
5. **Gradient Accumulation in Practice**：Hugging Face `Accelerate` 库的梯度累积文档，实用工程参考。
6. **ZeRO 优化器**（DeepSpeed）：进一步分片优化器状态、梯度、参数，是 DDP 的进阶形式。

## 关联练习 Related Assignments

```bash
python check.py w10-grad-accum
```

> 实现 `accumulate_and_step(model, optimizer, microbatches, loss_fn)`：
> 关键是每个 microbatch 的损失除以 `len(microbatches)` 再 `.backward()`，最后统一 `optimizer.step()`。

> 能力标签：**GA4** · threshold ≥ 0.7